# 01 — Feature extraction (`feature_extraction.py`)

AlphaFold's most important input is the **MSA**: sequences evolutionarily related to the query protein. This notebook follows the compact path from raw `.a3m` text to model-ready tensors. Each operation maps to one function in `../src/af2_from_scratch/feature_extraction.py`.

## Stage map

```text
A3M alignment
     |
     v
+----------------------+     +----------------------+
| Parse sequences      | --> | Remove insertions    |
| query + homologs     |     | keep deletion counts|
+----------------------+     +----------+-----------+
                                        |
                                        v
+----------------------+     +----------------------+
| Compute MSA profile  | <-- | Deduplicate + one-hot|
+----------+-----------+     +----------------------+
           |
           v
+----------------------+     +----------------------+
| Sample MSA rows      | --> | Mask training rows   |
+----------+-----------+     +----------+-----------+
           |                            |
           +-------------+--------------+
                         v
      msa_feat | extra_msa_feat | target_feat | residue_index
```

**Read alongside:** `../src/af2_from_scratch/feature_extraction.py`. The reusable boundary is `msa_features(...)` for parsing and `sample_batch(...)` for per-step augmentation.

In [ ]:
import sys
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, "../src")  # package source lives one level up
torch.manual_seed(0)
plt.rcParams["figure.figsize"] = (8, 4)

## 1. Look at the raw file
`>` lines are headers; the first sequence is the **query**. Uppercase = aligned residue, lowercase = insertion (to be removed, but *counted*), `-` = gap.

In [ ]:
lines = open("../examples/tautomerase/alignment.a3m").readlines()
print("".join(lines[:4]))

## 2. Parse one sequence
`parse_seq` uses a capture-group regex split: even indices are aligned chars, odd indices are lowercase insertion runs. A cumsum turns run lengths into 'insertions seen so far' = the **deletion count** per kept residue.

In [ ]:
from af2_from_scratch.feature_extraction import parse_seq

raw = "PIAxqIHsdILEGR"  # toy example: xq and sd are insertions
clean, dels = parse_seq(raw)
print("raw:     ", raw)
print("clean:   ", clean)  # insertions stripped
print("deletions:", dels)  # 0,0,0, then 2 after 'xq', then 2 more after 'sd'

## 3. Run the full extraction
`msa_features` parses all 8,361 sequences, dedups (7,934 unique), one-hot encodes (22 classes incl. gap), and computes the per-position **profile** = mean over the sequence axis (a dimension reduction!).

In [ ]:
from af2_from_scratch.feature_extraction import msa_features

f = msa_features("../examples/tautomerase/alignment.a3m")
for k, v in f.items():
    print(f"{k:16s} {tuple(v.shape)}")

The **profile** is the MSA compressed to its essence: which amino acid appears how often at each position. Conserved positions (sharp peaks) are usually structurally important.

In [ ]:
plt.imshow(f["profile"][:, :20].T, aspect="auto", cmap="viridis")
plt.xlabel("residue position")
plt.ylabel("amino acid (A..V)")
plt.title("MSA profile: evolution's per-position fingerprint")
plt.colorbar()
plt.show()

## 4. Training-time data: `sample_batch`
Two ideas in one function:
* **random subsample** of MSA rows (128 cluster + 128 extra) — replaces AF2's clustering *and* gives a fresh training view every step (data augmentation)
* **BERT-style masking** of cluster rows (15%): 70% zero-token / 10% random AA / 10% profile / 10% keep — the model must infer residues from context, exactly like masked-language-modeling

In [ ]:
from af2_from_scratch.feature_extraction import sample_batch

b = sample_batch(f, n_clu=128, n_ext=128, seed=0)
for k, v in b.items():
    print(f"{k:16s} {tuple(v.shape)}")
print("\nmsa_feat 45 = masked one-hot(22) + deletion(1) + profile(22)")
print("extra   23 = one-hot(22) + deletion(1)   (no profile -> simpler)")

**Recap:** raw evolutionary text → cleaned, deduplicated, one-hot tensors → augmented model batches. Next: `02_feature_embedding.ipynb` creates the MSA, pair, and extra-MSA representations.